# APEX360 VitalEdge -- AWS Deployment Notebook
## AI Surgical Robotics Commercial Intelligence Platform

Run every cell top to bottom once. The entire stack builds itself in ~15 minutes.

---

### What Gets Built

| AWS Service | What It Does | Monthly Cost |
|---|---|---|
| S3 (3 buckets) | Data lake + frontend hosting + KB docs | ~$0.01 |
| Lambda (7 functions) | Serverless API + KB auto-sync | ~$0.00 |
| API Gateway | REST API, 6 endpoints + agent routes | ~$1 |
| Cognito | Auth for 16 reps + KAM + AVP | ~$0 |
| Bedrock Claude | 3 AI agents (Surgical, Pricing, Travel) | ~$15 |
| OpenSearch (optional) | Knowledge base vector store | ~$90 |
| **Total without KB** | | **~$16/month** |
| **Total with KB** | | **~$106/month** |

---

### Folder Setup (Before You Run)

```
vitaledge-deploy/
  APEX360_VitalEdge_Deploy.ipynb   <- this notebook
  apex360_vitaledge.html           <- dashboard prototype
  csvs/
    accounts.csv
    activities.csv
    contacts.csv
    contracts.csv
    def_market_intelligence.csv
    device_models.csv
    discount_matrix.csv
    gpo_contracts.csv
    installed_base.csv
    opportunities.csv
    product_catalog.csv
    territories.csv
    win_loss.csv
```

### Prerequisites

```bash
pip install boto3 jupyter
aws configure          # Access Key + Secret + region (or use SageMaker -- no config needed)
jupyter notebook       # Launch from inside vitaledge-deploy/
```

### SageMaker Option (Recommended)
No `aws configure` needed -- SageMaker inherits IAM role automatically.
```
AWS Console -> SageMaker -> Notebook Instances -> Create
  Name:     apex360-deploy
  Instance: ml.t3.medium  (~$0.046/hr)
  IAM role: Create new role with AdministratorAccess
-> Upload this notebook + dashboard HTML + csvs/ folder
-> Run all cells
-> STOP the instance immediately after Cell 12
```


---
## Cell 0 -- Configuration

**Edit these five values before running anything else.**
Everything else auto-configures from these.


In [ ]:
import boto3, json, os, time, subprocess, io, zipfile, csv, urllib.request, re

# ============================================================
#  EDIT THESE BEFORE RUNNING
# ============================================================
CONFIG = {
    # AWS region -- must match your SageMaker instance region
    "region": "us-east-1",

    # S3 bucket names -- globally unique across ALL of AWS
    # Add your initials + a number: "apex360-data-jws42"
    "data_bucket":    "apex360-data-YOURSUFFIX",
    "app_bucket":     "apex360-app-YOURSUFFIX",
    "kb_docs_bucket": "apex360-kb-YOURSUFFIX",

    # Your email -- you get the AVP login
    "admin_email": "your.email@gmail.com",

    # Paths (relative to this notebook)
    "csv_folder": "./csvs",
    "html_file":  "./apex360_vitaledge.html",

    # ---- Do not edit below this line ----
    "lambda_role_name": "apex360-lambda-role",
    "kb_role_name":     "apex360-bedrock-kb-role",
    "cognito_pool_name":"apex360-users",
    "account_id": "", "lambda_role_arn": "", "kb_role_arn": "",
    "api_id": "", "api_url": "", "app_url": "",
    "cognito_pool_id": "", "cognito_client_id": "",
    "kb_ids": {}
}
# ============================================================

sts = boto3.client("sts", region_name=CONFIG["region"])
CONFIG["account_id"] = sts.get_caller_identity()["Account"]

print("=" * 62)
print("  APEX360 VitalEdge -- Deployment Configuration")
print("=" * 62)
print(f"  AWS Account:  {CONFIG['account_id']}")
print(f"  Region:       {CONFIG['region']}")
print(f"  Data bucket:  {CONFIG['data_bucket']}")
print(f"  App bucket:   {CONFIG['app_bucket']}")
print(f"  Admin email:  {CONFIG['admin_email']}")
print("=" * 62)

# Verify CSV folder
print()
if os.path.exists(CONFIG["csv_folder"]):
    csvs = [f for f in os.listdir(CONFIG["csv_folder"]) if f.endswith(".csv")]
    print(f"  CSV files found: {len(csvs)}/13")
    for c in sorted(csvs):
        rows = sum(1 for _ in open(os.path.join(CONFIG["csv_folder"], c))) - 1
        print(f"    {c:<45} {rows:>4} rows")
    if len(csvs) < 13:
        print()
        print("  WARNING: Expected 13 CSVs. Check your csvs/ folder.")
else:
    print(f"  WARNING: csvs/ folder not found at {CONFIG['csv_folder']}")
    print("  Create the folder and add all 13 CSVs before running Cell 3.")

# Verify HTML
print()
if os.path.exists(CONFIG["html_file"]):
    kb = os.path.getsize(CONFIG["html_file"]) // 1024
    print(f"  Dashboard: {CONFIG['html_file']} ({kb} KB) -- ready")
else:
    print(f"  WARNING: Dashboard not found at {CONFIG['html_file']}")


---
## Cell 1 -- IAM Role for Lambda

Creates `apex360-lambda-role` -- the identity badge all Lambda functions run under.
Grants: CloudWatch logs + S3 read + Bedrock full access.
The 12-second pause lets IAM propagate globally before Lambda tries to use the role.


In [ ]:
iam = boto3.client("iam", region_name=CONFIG["region"])

trust = {
    "Version": "2012-10-17",
    "Statement": [{"Effect": "Allow",
        "Principal": {"Service": "lambda.amazonaws.com"},
        "Action": "sts:AssumeRole"}]
}

try:
    r = iam.create_role(RoleName=CONFIG["lambda_role_name"],
                        AssumeRolePolicyDocument=json.dumps(trust),
                        Description="APEX360 VitalEdge Lambda execution role")
    CONFIG["lambda_role_arn"] = r["Role"]["Arn"]
    print(f"  Created: {CONFIG['lambda_role_arn']}")
except iam.exceptions.EntityAlreadyExistsException:
    CONFIG["lambda_role_arn"] = iam.get_role(
        RoleName=CONFIG["lambda_role_name"])["Role"]["Arn"]
    print(f"  Exists:  {CONFIG['lambda_role_arn']}")

for policy in [
    "arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole",
    "arn:aws:iam::aws:policy/AmazonS3ReadOnlyAccess",
    "arn:aws:iam::aws:policy/AmazonBedrockFullAccess",
]:
    try:
        iam.attach_role_policy(RoleName=CONFIG["lambda_role_name"], PolicyArn=policy)
        print(f"  Attached: {policy.split('/')[-1]}")
    except: pass

print()
print("  Waiting 12s for IAM global propagation...")
time.sleep(12)
print("  IAM role ready.")


---
## Cell 2 -- Create S3 Buckets

Creates three buckets:

| Bucket | Access | Purpose |
|---|---|---|
| `apex360-data-*` | Private | CSV data lake |
| `apex360-app-*` | Public read | Static website (serves the dashboard) |
| `apex360-kb-*` | Private | Bedrock KB document store |

Also creates folder structure in the KB bucket:
- `surgical/` -- APEX product specs, clinical evidence, competitive battlecards
- `pricing/` -- GPO contracts, discount matrix, ROI models
- `travel/` -- territory guidelines, account priorities


In [ ]:
s3 = boto3.client("s3", region_name=CONFIG["region"])

def make_bucket(name, public=False):
    try:
        if CONFIG["region"] == "us-east-1":
            s3.create_bucket(Bucket=name)
        else:
            s3.create_bucket(Bucket=name,
                CreateBucketConfiguration={"LocationConstraint": CONFIG["region"]})
        print(f"  Created: s3://{name}")
    except s3.exceptions.BucketAlreadyOwnedByYou:
        print(f"  Exists:  s3://{name}")
    except Exception as e:
        print(f"  ERROR:   {name} -- {e}")
        raise

    if public:
        # Remove AWS default "block all public access" guardrail
        s3.put_public_access_block(Bucket=name,
            PublicAccessBlockConfiguration={
                "BlockPublicAcls": False, "IgnorePublicAcls": False,
                "BlockPublicPolicy": False, "RestrictPublicBuckets": False})
        # Grant public GET so any browser can load the dashboard
        s3.put_bucket_policy(Bucket=name, Policy=json.dumps({
            "Version": "2012-10-17",
            "Statement": [{"Effect": "Allow", "Principal": "*",
                "Action": "s3:GetObject",
                "Resource": f"arn:aws:s3:::{name}/*"}]}))
        # Enable static website -- index.html fallback handles SPA routing
        s3.put_bucket_website(Bucket=name,
            WebsiteConfiguration={
                "IndexDocument": {"Suffix": "index.html"},
                "ErrorDocument": {"Key": "index.html"}})
        print(f"    Configured as public static website")

make_bucket(CONFIG["data_bucket"],    public=False)
make_bucket(CONFIG["app_bucket"],     public=True)
make_bucket(CONFIG["kb_docs_bucket"], public=False)

# KB folder structure -- helps the auto-sync Lambda route uploads to the right KB
for prefix in [
    "surgical/spec_sheets/",
    "surgical/clinical_evidence/",
    "surgical/competitive/",
    "surgical/training/",
    "pricing/gpo_contracts/",
    "pricing/guides/",
    "pricing/roi_models/",
    "travel/",
]:
    s3.put_object(Bucket=CONFIG["kb_docs_bucket"], Key=prefix, Body=b"")
    print(f"  KB folder: {prefix}")

print()
print(f"  Dashboard URL (available after Cell 8):")
print(f"  http://{CONFIG['app_bucket']}.s3-website-{CONFIG['region']}.amazonaws.com")


---
## Cell 3 -- Upload CSV Data to S3

Reads your 13 CSVs from `./csvs/` and uploads them to the data bucket.

**To update data later:** Replace any CSV in `./csvs/` and re-run this cell.
Lambda reads from S3 on every API call, so the dashboard updates immediately.
No Lambda or API changes needed.

**CSV to S3 key mapping:**
```
territories.csv          -> territories/territories.csv
accounts.csv             -> accounts/accounts.csv
contacts.csv             -> accounts/contacts.csv
opportunities.csv        -> pipeline/opportunities.csv
installed_base.csv       -> installed_base/installed_base.csv
device_models.csv        -> installed_base/device_models.csv
contracts.csv            -> contracts/contracts.csv
win_loss.csv             -> win_loss/win_loss.csv
activities.csv           -> activities/activities.csv
def_market_intelligence  -> market_intel/def_market_intelligence.csv
gpo_contracts.csv        -> reference/gpo_contracts.csv
product_catalog.csv      -> reference/product_catalog.csv
discount_matrix.csv      -> reference/discount_matrix.csv
```


In [ ]:
CSV_KEYS = {
    "territories.csv":              "territories/territories.csv",
    "accounts.csv":                 "accounts/accounts.csv",
    "contacts.csv":                 "accounts/contacts.csv",
    "opportunities.csv":            "pipeline/opportunities.csv",
    "installed_base.csv":           "installed_base/installed_base.csv",
    "device_models.csv":            "installed_base/device_models.csv",
    "contracts.csv":                "contracts/contracts.csv",
    "win_loss.csv":                 "win_loss/win_loss.csv",
    "activities.csv":               "activities/activities.csv",
    "def_market_intelligence.csv":  "market_intel/def_market_intelligence.csv",
    "gpo_contracts.csv":            "reference/gpo_contracts.csv",
    "product_catalog.csv":          "reference/product_catalog.csv",
    "discount_matrix.csv":          "reference/discount_matrix.csv",
}

print(f"Uploading to s3://{CONFIG['data_bucket']}/")
print()

total_rows = 0
missing = []
for filename, s3_key in CSV_KEYS.items():
    local = os.path.join(CONFIG["csv_folder"], filename)
    if not os.path.exists(local):
        print(f"  MISSING  {filename}")
        missing.append(filename)
        continue
    with open(local, "r", encoding="utf-8") as f:
        content = f.read()
    rows = content.count("\n") - 1
    s3.put_object(Bucket=CONFIG["data_bucket"], Key=s3_key,
                  Body=content.encode("utf-8"), ContentType="text/csv")
    total_rows += rows
    print(f"  OK  {filename:<45} {rows:>4} rows")

print()
print(f"  Total rows uploaded: {total_rows:,}")
if missing:
    print(f"  Missing files: {missing}")
    print("  Add missing CSVs to ./csvs/ and re-run this cell.")
else:
    print("  All 13 CSVs uploaded successfully.")


---
## Cell 4 -- Deploy Lambda Functions

Deploys 6 Lambda functions -- one per data domain:

| Function | Reads | Returns |
|---|---|---|
| `apex360-pipeline` | opportunities.csv | Deals filtered by territory |
| `apex360-accounts` | accounts.csv + installed_base.csv | Account profiles with device summary |
| `apex360-territories` | territories.csv + opportunities.csv | Territories with live pipeline rollup |
| `apex360-installed-base` | installed_base.csv + device_models.csv | Device inventory joined with model data |
| `apex360-win-loss` | win_loss.csv | W/L records with verbatims |
| `apex360-agent` | (calls Bedrock) | Claude response for Surgical/Pricing/Travel agent |

**Territory security:** Every function extracts `territory_id` from the Cognito JWT.
Reps see only their territory. You (AVP) see everything.


In [ ]:
lam = boto3.client("lambda", region_name=CONFIG["region"])

def deploy_lambda(name, source):
    # Package source code as ZIP and create/update a Lambda function.
    buf = io.BytesIO()
    with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.writestr("handler.py", source)
    buf.seek(0)
    kw = dict(
        FunctionName=name, Runtime="python3.11",
        Role=CONFIG["lambda_role_arn"], Handler="handler.handler",
        Code={"ZipFile": buf.read()}, Timeout=30, MemorySize=256,
        Environment={"Variables": {
            "DATA_BUCKET": CONFIG["data_bucket"],
            "REGION":      CONFIG["region"]
        }}
    )
    try:
        lam.create_function(**kw)
        print(f"  Created:  {name}")
    except lam.exceptions.ResourceConflictException:
        buf.seek(0)
        lam.update_function_code(FunctionName=name, ZipFile=buf.read())
        print(f"  Updated:  {name}")

# Shared boilerplate -- prepended to every Lambda
# Provides: S3 client, CORS headers, ok() builder, read_csv(), get_identity(), is_avp()
SHARED = (
    "import boto3,csv,io,json,os\n"
    "s3=boto3.client('s3')\n"
    "B=os.environ['DATA_BUCKET']\n"
    "CORS={'Content-Type':'application/json',"
    "'Access-Control-Allow-Origin':'*',"
    "'Access-Control-Allow-Methods':'GET,POST,OPTIONS',"
    "'Access-Control-Allow-Headers':'Content-Type,Authorization'}\n"
    "def ok(b): return {'statusCode':200,'headers':CORS,'body':json.dumps(b)}\n"
    "def read_csv(k):\n"
    "    obj=s3.get_object(Bucket=B,Key=k)\n"
    "    return list(csv.DictReader(io.StringIO(obj['Body'].read().decode())))\n"
    # JWT claims injected by API Gateway Cognito authorizer
    "def get_id(e):\n"
    "    c=(e.get('requestContext',{}).get('authorizer',{}).get('claims',{}))\n"
    "    return {'tid':c.get('custom:territory_id','ALL'),"
    "'role':c.get('custom:role','rep'),"
    "'email':c.get('email',''),"
    "'name':c.get('custom:rep_name','')}\n"
    # AVP, RVP, admin see all territories; reps are restricted to their own
    "def avp(i): return i['role'] in ('avp','rvp','admin')\n"
)

# Pipeline Lambda
deploy_lambda("apex360-pipeline", SHARED +
"def handler(e,c):\n"
"  if (e.get('httpMethod') or '')==='OPTIONS': return ok({})\n"
"  i=get_id(e); q=e.get('queryStringParameters') or {}\n"
"  tid=q.get('territory','ALL') if avp(i) else i['tid']\n"
"  rows=read_csv('pipeline/opportunities.csv')\n"
"  deals=[]\n"
"  for r in rows:\n"
"    if tid!='ALL' and r.get('territory_id')!=tid: continue\n"
"    try:\n"
"      deals.append({'id':r['opportunity_id'],'name':r['opportunity_name'],\n"
"        'account':r['account_id'],'territory':r['territory_id'],\n"
"        'rep':r['rep_name'],'amount':float(r['amount']),\n"
"        'weighted':float(r['weighted_amount']),\n"
"        'stage':r['stage'],'probability':int(r['probability']),\n"
"        'closeDate':r['close_date'],'laer':r['laer_stage'],\n"
"        'model':r['commercial_model'],'gpo':r.get('gpo_applicable',''),\n"
"        'risk':r['risk_level'],'forecast':r['forecast_category'],\n"
"        'nextAction':r['next_action'],'actionDate':r['action_due_date'],\n"
"        'q3':r.get('is_q3','FALSE')=='TRUE'})\n"
"    except: pass\n"
"  deals.sort(key=lambda x:x['amount'],reverse=True)\n"
"  return ok({'deals':deals,'count':len(deals),\n"
"    'total':sum(d['amount'] for d in deals),\n"
"    'weighted':sum(d['weighted'] for d in deals)})\n"
)

# Accounts Lambda -- joins installed base for device summary
deploy_lambda("apex360-accounts", SHARED +
"def handler(e,c):\n"
"  if (e.get('httpMethod') or '')==='OPTIONS': return ok({})\n"
"  i=get_id(e); q=e.get('queryStringParameters') or {}\n"
"  aid=q.get('id')\n"
"  rows=read_csv('accounts/accounts.csv')\n"
"  if not avp(i): rows=[r for r in rows if r.get('territory')==i['tid']]\n"
"  if aid: rows=[r for r in rows if r['account_id']==aid]\n"
"  try:\n"
"    ib=read_csv('installed_base/installed_base.csv')\n"
"    ib_sum={}\n"
"    for b in ib:\n"
"      a=b['account_id']\n"
"      if a not in ib_sum: ib_sum[a]={'total':0,'competitive':0,'eol':0,'arv':0}\n"
"      q2=int(b.get('quantity',1))\n"
"      ib_sum[a]['total']+=q2\n"
"      if b.get('status')=='Competitive': ib_sum[a]['competitive']+=q2\n"
"      if 'EOL' in b.get('status',''): ib_sum[a]['eol']+=q2\n"
"      try: ib_sum[a]['arv']+=float(b.get('annual_service_value',0))\n"
"      except: pass\n"
"  except: ib_sum={}\n"
"  accts=[]\n"
"  for r in rows:\n"
"    try:\n"
"      ib=ib_sum.get(r['account_id'],{})\n"
"      accts.append({'id':r['account_id'],'name':r['account_name'],\n"
"        'type':r['idn_type'],'hospitals':int(r.get('hospitals',0)),\n"
"        'hqCity':r.get('hq_city',''),'ehr':r.get('ehr_system',''),\n"
"        'beds':int(r.get('total_beds',0)),\n"
"        'apexInstalled':float(r.get('apex_installed',0)),\n"
"        'competitive':float(r.get('competitive_installed',0)),\n"
"        'whitespace':float(r.get('whitespace',0)),\n"
"        'definitiveScore':int(r.get('definitive_score',0)),\n"
"        'gpo':r.get('gpo',''),'territory':r.get('territory',''),\n"
"        'rep':r.get('rep_name',''),\n"
"        'dealAmount':float(r.get('deal_amount',0)),\n"
"        'dealStatus':r.get('deal_status',''),\n"
"        'totalSystems':ib.get('total',0),\n"
"        'competitiveSystems':ib.get('competitive',0),\n"
"        'eolSystems':ib.get('eol',0),\n"
"        'annualServiceValue':ib.get('arv',0)})\n"
"    except: pass\n"
"  return ok({'accounts':accts,'count':len(accts)})\n"
)

# Territories Lambda -- includes live pipeline rollup from opportunities
deploy_lambda("apex360-territories", SHARED +
"def handler(e,c):\n"
"  if (e.get('httpMethod') or '')==='OPTIONS': return ok({})\n"
"  i=get_id(e)\n"
"  rows=read_csv('territories/territories.csv')\n"
"  if not avp(i): rows=[r for r in rows if r['territory_id']==i['tid']]\n"
"  try:\n"
"    opps=read_csv('pipeline/opportunities.csv')\n"
"    pt={}\n"
"    for o in opps:\n"
"      t=o['territory_id']\n"
"      if t not in pt: pt[t]={'total':0,'wtd':0,'count':0,'q3':0,'commit':0}\n"
"      amt=float(o.get('amount',0)); wtd=float(o.get('weighted_amount',0))\n"
"      pt[t]['total']+=amt; pt[t]['wtd']+=wtd; pt[t]['count']+=1\n"
"      if o.get('is_q3')=='TRUE': pt[t]['q3']+=1\n"
"      if o.get('forecast_category')=='Commit': pt[t]['commit']+=amt\n"
"  except: pt={}\n"
"  terrs=[]\n"
"  for r in rows:\n"
"    tid=r['territory_id']; p=pt.get(tid,{})\n"
"    terrs.append({'id':tid,'rep':r['rep_name'],'email':r.get('rep_email',''),\n"
"      'states':r['states'].split('|'),'region':r.get('region',''),\n"
"      'pipeline':float(r['pipeline_m']),'op':float(r['op_target_m']),\n"
"      'color':r['color'],'attainment':int(r['attainment_pct']),\n"
"      'livePipeline':round(p.get('total',0)/1e6,2),\n"
"      'liveWeighted':round(p.get('wtd',0)/1e6,2),\n"
"      'dealCount':p.get('count',0),'q3Count':p.get('q3',0),\n"
"      'commitValue':round(p.get('commit',0)/1e6,2)})\n"
"  return ok({'territories':terrs})\n"
)

# Installed Base Lambda -- joins device_models for full context
deploy_lambda("apex360-installed-base", SHARED +
"def handler(e,c):\n"
"  if (e.get('httpMethod') or '')==='OPTIONS': return ok({})\n"
"  q=e.get('queryStringParameters') or {}\n"
"  aid=q.get('account_id','ALL')\n"
"  rows=read_csv('installed_base/installed_base.csv')\n"
"  if aid!='ALL': rows=[r for r in rows if r['account_id']==aid]\n"
"  try:\n"
"    models={m['model_id']:m for m in read_csv('installed_base/device_models.csv')}\n"
"  except: models={}\n"
"  devices=[]\n"
"  for r in rows:\n"
"    try:\n"
"      m=models.get(r.get('model_id',''),{})\n"
"      devices.append({'installId':r['install_id'],'accountId':r['account_id'],\n"
"        'modelName':m.get('model_name',r.get('model_id','')),\n"
"        'manufacturer':m.get('manufacturer',''),\n"
"        'productLine':m.get('product_line',''),\n"
"        'quantity':int(r.get('quantity',1)),'dept':r.get('department',''),\n"
"        'age':float(r.get('age_years',0)),\n"
"        'apexNative':m.get('apex_native','FALSE')=='TRUE',\n"
"        'status':r['status'],'note':r.get('sales_note',''),\n"
"        'priority':r.get('displacement_priority',''),\n"
"        'contractExpiry':r.get('contract_expiry',''),\n"
"        'annualServiceValue':float(r.get('annual_service_value',0))})\n"
"    except: pass\n"
"  devices.sort(key=lambda x:x['annualServiceValue'],reverse=True)\n"
"  return ok({'devices':devices,'count':len(devices),\n"
"    'totalARV':sum(d['annualServiceValue'] for d in devices),\n"
"    'competitive':sum(1 for d in devices if d['status']=='Competitive'),\n"
"    'eol':sum(1 for d in devices if 'EOL' in d['status'])})\n"
)

# Win/Loss Lambda
deploy_lambda("apex360-win-loss", SHARED +
"def handler(e,c):\n"
"  if (e.get('httpMethod') or '')==='OPTIONS': return ok({})\n"
"  i=get_id(e); q=e.get('queryStringParameters') or {}\n"
"  outcome=q.get('outcome','ALL')\n"
"  competitor=q.get('competitor','ALL')\n"
"  rows=read_csv('win_loss/win_loss.csv')\n"
"  if not avp(i): rows=[r for r in rows if r.get('territory_id')==i['tid']]\n"
"  if outcome!='ALL': rows=[r for r in rows if r['outcome']==outcome.upper()]\n"
"  if competitor!='ALL': rows=[r for r in rows if competitor.lower() in r.get('competitor','').lower()]\n"
"  results=[]\n"
"  for r in rows:\n"
"    try:\n"
"      results.append({'date':r['date'],'accountId':r['account_id'],\n"
"        'accountName':r['account_name'],'rep':r['rep_name'],\n"
"        'amount':float(r['amount']),'productLine':r['product_line'],\n"
"        'model':r['commercial_model'],'competitor':r['competitor'],\n"
"        'outcome':r['outcome'],'reason':r['primary_reason'],\n"
"        'quote':r['customer_quote'],'territory':r.get('territory_id',''),\n"
"        'execSponsor':r.get('exec_sponsor_engaged','FALSE')=='TRUE',\n"
"        'gpoPricing':r.get('gpo_pricing_used','FALSE')=='TRUE',\n"
"        'lostToIncumbent':r.get('lost_to_incumbent','FALSE')=='TRUE',\n"
"        'priceWasFactor':r.get('price_was_factor','FALSE')=='TRUE'})\n"
"    except: pass\n"
"  results.sort(key=lambda x:x['date'],reverse=True)\n"
"  wins=sum(1 for r in results if r['outcome']=='WIN')\n"
"  losses=sum(1 for r in results if r['outcome']=='LOSS')\n"
"  return ok({'results':results,'count':len(results),\n"
"    'wins':wins,'losses':losses,\n"
"    'winRate':round(wins/max(wins+losses,1)*100,1),\n"
"    'winRevenue':sum(r['amount'] for r in results if r['outcome']=='WIN'),\n"
"    'lossRevenue':sum(r['amount'] for r in results if r['outcome']=='LOSS')})\n"
)

# Agent Lambda -- Bedrock Claude with VitalEdge system prompts
SYSTEMS = {
    "surgical": (
        "You are the APEX360 Surgical Intelligence Agent for a VitalEdge Area VP managing 15 reps. "
        "VitalEdge makes APEX -- an AI-powered surgical robotics and interventional imaging platform. "
        "You have deep expertise in robotic surgery, APEX AI surgical guidance, complication rate reduction, "
        "case volume ROI, remote proctoring, OR workflow optimization, and competitive positioning "
        "vs Intuitive Surgical da Vinci, Stryker Mako, and Medtronic Hugo. "
        "Answer in 3-5 sentences. Use bullets for comparisons. "
        "Always connect answers to how a rep can use this to win a deal or advance an opportunity."
    ),
    "pricing": (
        "You are the APEX360 Contract and Pricing Agent for VitalEdge. "
        "You know GPO pricing (Vizient ceiling $2.4M, Premier $2.2M, HealthTrust $2.6M per system), "
        "commercial models (Capital+Subscription, RaaS at $38K/month, Per-Procedure at $2,800/case), "
        "discount approval levels (Field Rep 8%, AVP 14%, RVP 20%, VP Sales 26%), "
        "and can build 5-year TCO vs Intuitive da Vinci comparisons. "
        "Answer with specific dollar amounts. Build ROI models when asked. "
        "Always state which approval level a proposed discount requires."
    ),
    "travel": (
        "You are the APEX360 Travel Agent for a VitalEdge Area VP managing 15 reps. "
        "Optimize travel for maximum customer-facing time with Chiefs of Surgery, "
        "VP Surgical Services, CMOs, and CFOs. "
        "Prioritize: at-risk deals over $1M first, then expansion accounts, "
        "then EOL displacement opportunities, then rep coaching. "
        "Generate specific day-by-day itineraries with account, time, objective, "
        "and who to bring (Solutions Engineer, Clinical Education Specialist, AVP)."
    ),
}

agent_src = (
    "import boto3,json,os\n"
    "bedrock=boto3.client('bedrock-runtime',region_name=os.environ['REGION'])\n"
    "CORS={'Content-Type':'application/json','Access-Control-Allow-Origin':'*',"
    "'Access-Control-Allow-Methods':'GET,POST,OPTIONS',"
    "'Access-Control-Allow-Headers':'Content-Type,Authorization'}\n"
    "SYSTEMS=" + repr(SYSTEMS) + "\n"
    "def handler(e,c):\n"
    "  if (e.get('httpMethod') or '')==='OPTIONS':\n"
    "    return {'statusCode':200,'headers':CORS,'body':''}\n"
    "  atype=(e.get('pathParameters') or {}).get('type','surgical')\n"
    "  body=json.loads(e.get('body') or '{}')\n"
    "  message=body.get('message','')\n"
    "  history=body.get('history',[])\n"
    "  messages=history+[{'role':'user','content':message}]\n"
    "  resp=bedrock.invoke_model(\n"
    "    modelId='anthropic.claude-3-5-sonnet-20241022-v2:0',\n"
    "    body=json.dumps({'anthropic_version':'bedrock-2023-05-31',\n"
    "      'max_tokens':1500,\n"
    "      'system':SYSTEMS.get(atype,SYSTEMS['surgical']),\n"
    "      'messages':messages}))\n"
    "  result=json.loads(resp['body'].read())\n"
    "  answer=result['content'][0]['text']\n"
    "  return {'statusCode':200,'headers':CORS,\n"
    "    'body':json.dumps({'response':answer,'agentType':atype})}\n"
)
deploy_lambda("apex360-agent", agent_src)

time.sleep(5)
print()
print("All 6 Lambda functions deployed.")
print("View: AWS Console -> Lambda -> filter 'apex360'")


---
## Cell 5 -- Create API Gateway

Creates a REST API with 6 routes and wires each to its Lambda function.
OPTIONS methods are added automatically on every route for CORS preflight.

**Routes created:**
- `GET /pipeline` -- supports `?territory=NW-1`
- `GET /accounts` -- supports `?id=ACC-001`
- `GET /territories`
- `GET /installed-base` -- supports `?account_id=ACC-001`
- `GET /win-loss` -- supports `?outcome=WIN&competitor=Intuitive`
- `POST /agent/{type}` -- type = surgical / pricing / travel

**Final URL format:**
`https://{id}.execute-api.{region}.amazonaws.com/prod`

This URL gets injected into the dashboard HTML in Cell 7.


In [ ]:
apigw = boto3.client("apigateway", region_name=CONFIG["region"])

api    = apigw.create_rest_api(name="apex360-api",
             description="APEX360 VitalEdge data and AI agent API",
             endpointConfiguration={"types": ["REGIONAL"]})
api_id = api["id"]
print(f"  API created: {api_id}")

root_id = [r for r in apigw.get_resources(restApiId=api_id)["items"]
           if r["path"] == "/"][0]["id"]

def add_route(path, fn_name, methods=None, parent=None):
    if methods is None: methods = ["GET"]
    parent = parent or root_id
    res_id = apigw.create_resource(
        restApiId=api_id, parentId=parent, pathPart=path)["id"]
    fn_arn = lam.get_function(FunctionName=fn_name)["Configuration"]["FunctionArn"]
    uri = (f"arn:aws:apigateway:{CONFIG['region']}:lambda:"
           f"path/2015-03-31/functions/{fn_arn}/invocations")
    for method in methods + ["OPTIONS"]:
        apigw.put_method(restApiId=api_id, resourceId=res_id,
                         httpMethod=method, authorizationType="NONE")
        apigw.put_integration(restApiId=api_id, resourceId=res_id,
                              httpMethod=method, type="AWS_PROXY",
                              integrationHttpMethod="POST", uri=uri)
    try:
        lam.add_permission(
            FunctionName=fn_name,
            StatementId=f"apigw-{path.replace('{','').replace('}','')}",
            Action="lambda:InvokeFunction",
            Principal="apigateway.amazonaws.com",
            SourceArn=(f"arn:aws:execute-api:{CONFIG['region']}:"
                       f"{CONFIG['account_id']}:{api_id}/*/*"))
    except: pass
    print(f"  Route: /{path}  ->  {fn_name}")
    return res_id

add_route("pipeline",       "apex360-pipeline")
add_route("accounts",       "apex360-accounts")
add_route("territories",    "apex360-territories")
add_route("installed-base", "apex360-installed-base")
add_route("win-loss",       "apex360-win-loss")

# /agent/{type} -- two-level path
agent_parent = apigw.create_resource(
    restApiId=api_id, parentId=root_id, pathPart="agent")["id"]
add_route("{type}", "apex360-agent", methods=["POST"], parent=agent_parent)

apigw.create_deployment(restApiId=api_id, stageName="prod",
                        description="APEX360 initial deployment")

CONFIG["api_id"]  = api_id
CONFIG["api_url"] = (f"https://{api_id}.execute-api"
                     f".{CONFIG['region']}.amazonaws.com/prod")

print()
print(f"  API live: {CONFIG['api_url']}")
print(f"  Test:     curl {CONFIG['api_url']}/territories")


---
## Cell 6 -- Cognito Authentication

Creates a Cognito User Pool with 17 user accounts (15 reps + 1 KAM + 1 AVP).

**How territory security works:**
1. Rep logs in -> Cognito issues JWT token containing `custom:territory_id = NW-1`
2. React sends JWT as `Authorization` header on every API call
3. API Gateway validates JWT signature against this user pool
4. Decoded claims pass to Lambda as `event.requestContext.authorizer.claims`
5. Lambda reads `territory_id` and filters -- rep cannot see another territory

**Your AVP login:**
- Email: `your admin_email` (set in Cell 0)
- Password: `APEX360!Avp2025` (change on first login)
- Access: ALL territories

**Cost:** $0 for up to 50,000 monthly active users.


In [ ]:
cognito = boto3.client("cognito-idp", region_name=CONFIG["region"])

pool = cognito.create_user_pool(
    PoolName=CONFIG["cognito_pool_name"],
    Policies={"PasswordPolicy": {
        "MinimumLength": 8, "RequireUppercase": True,
        "RequireNumbers": True, "RequireSymbols": False,
        "TemporaryPasswordValidityDays": 30
    }},
    Schema=[
        {"Name": "territory_id", "AttributeDataType": "String", "Mutable": True},
        {"Name": "role",         "AttributeDataType": "String", "Mutable": True},
        {"Name": "rep_name",     "AttributeDataType": "String", "Mutable": True},
    ],
    AutoVerifiedAttributes=["email"],
    MfaConfiguration="OFF",
)
pool_id = pool["UserPool"]["Id"]
CONFIG["cognito_pool_id"] = pool_id
print(f"  User pool: {pool_id}")

client = cognito.create_user_pool_client(
    UserPoolId=pool_id,
    ClientName="apex360-web",
    GenerateSecret=False,
    ExplicitAuthFlows=[
        "ALLOW_USER_PASSWORD_AUTH",
        "ALLOW_REFRESH_TOKEN_AUTH",
        "ALLOW_USER_SRP_AUTH",
    ],
    ReadAttributes=["email","name",
                    "custom:territory_id","custom:role","custom:rep_name"],
    TokenValidityUnits={"AccessToken":"hours","IdToken":"hours","RefreshToken":"days"},
    AccessTokenValidity=8, IdTokenValidity=8, RefreshTokenValidity=30,
)
CONFIG["cognito_client_id"] = client["UserPoolClient"]["ClientId"]
print(f"  App client: {CONFIG['cognito_client_id']}")

# 15 reps + 1 KAM + 1 AVP
# Format: (email, display_name, territory_id, role, temp_password)
USERS = [
    (CONFIG["admin_email"],         "AVP User",           "ALL",  "avp", "APEX360!Avp2025"),
    ("s.chen@vitaledge.com",        "Dr. Sarah Chen",     "NW-1", "rep", "APEX360!Nw12025"),
    ("m.webb@vitaledge.com",        "Marcus Webb",        "NW-2", "rep", "APEX360!Nw22025"),
    ("p.anand@vitaledge.com",       "Priya Anand",        "SW-1", "rep", "APEX360!Sw12025"),
    ("c.reyes@vitaledge.com",       "Carlos Reyes",       "SW-2", "rep", "APEX360!Sw22025"),
    ("j.park@vitaledge.com",        "Jessica Park",       "SW-3", "rep", "APEX360!Sw32025"),
    ("t.okafor@vitaledge.com",      "Thomas Okafor",      "MW-1", "rep", "APEX360!Mw12025"),
    ("e.harrington@vitaledge.com",  "Emily Harrington",   "MW-2", "rep", "APEX360!Mw22025"),
    ("d.novak@vitaledge.com",       "Daniel Novak",       "MW-3", "rep", "APEX360!Mw32025"),
    ("a.brooks@vitaledge.com",      "Aaliyah Brooks",     "SE-1", "rep", "APEX360!Se12025"),
    ("r.castellano@vitaledge.com",  "Ryan Castellano",    "SE-2", "rep", "APEX360!Se22025"),
    ("s.delacroix@vitaledge.com",   "Sophia Delacroix",   "NE-1", "rep", "APEX360!Ne12025"),
    ("j.whitmore@vitaledge.com",    "James Whitmore",     "NE-2", "rep", "APEX360!Ne22025"),
    ("f.alrashid@vitaledge.com",    "Fatima Al-Rashid",   "NE-3", "rep", "APEX360!Ne32025"),
    ("k.zhao@vitaledge.com",        "Kevin Zhao",         "SC-1", "rep", "APEX360!Sc12025"),
    ("v.sterling@vitaledge.com",    "Victoria Sterling",  "KAM",  "kam", "APEX360!Kam2025"),
]

print()
for email, name, tid, role, pw in USERS:
    try:
        cognito.admin_create_user(
            UserPoolId=pool_id, Username=email,
            TemporaryPassword=pw,
            UserAttributes=[
                {"Name": "email",               "Value": email},
                {"Name": "name",                "Value": name},
                {"Name": "email_verified",      "Value": "true"},
                {"Name": "custom:territory_id", "Value": tid},
                {"Name": "custom:role",         "Value": role},
                {"Name": "custom:rep_name",     "Value": name},
            ],
            MessageAction="SUPPRESS",
        )
        suffix = " <-- AVP (all territories)" if tid == "ALL" else ""
        print(f"  {email:<42} [{tid:<5}] [{role}]{suffix}")
    except cognito.exceptions.UsernameExistsException:
        print(f"  Exists: {email}")

print()
print(f"  Pool ID:   {pool_id}")
print(f"  Client ID: {CONFIG['cognito_client_id']}")
print()
print(f"  AVP login: {CONFIG['admin_email']} / APEX360!Avp2025")


---
## Cell 7 -- Patch Dashboard HTML with Live API URL

Reads `apex360_vitaledge.html` and injects a script block that:

1. Sets `API_BASE` to your live API Gateway URL
2. Loads live territories from `/territories` and rebuilds the US map
3. Replaces hardcoded pipeline deals with live data from `/pipeline`
4. Loads live win/loss data from `/win-loss`
5. Routes all three AI agent chats to `/agent/surgical`, `/agent/pricing`, `/agent/travel`
6. Shows a green **"Live API"** badge in the dashboard header

The patched file is saved as `apex360_live.html`.
The original `apex360_vitaledge.html` is untouched.

**To redeploy after changes:** Edit the HTML, re-run Cell 7, then re-run Cell 8.


In [ ]:
import re

with open(CONFIG["html_file"], "r", encoding="utf-8") as f:
    html = f.read()

api_url = CONFIG["api_url"]
print(f"Patching: {CONFIG['html_file']} ({len(html):,} chars)")
print(f"API URL:  {api_url}")

# Build the live integration script as a single string
# Using triple-quote string avoids all the concatenation issues
live_script = """

// APEX360 Live API Integration -- auto-injected by deployment notebook
var API_BASE = \'__API_URL__\';

function apiFetch(path, opts) {
  opts = opts || {};
  opts.headers = opts.headers || {};
  opts.headers[\'Content-Type\'] = \'application/json\';
  var tk = Object.keys(localStorage).find(function(k){ return k.includes(\'idToken\'); });
  if (tk) opts.headers[\'Authorization\'] = localStorage.getItem(tk);
  return fetch(API_BASE + path, opts).then(function(r){ return r.json(); });
}

function loadTerritories() {
  apiFetch(\'/territories\').then(function(data) {
    if (!data.territories) return;
    data.territories.forEach(function(t) {
      TERRITORY_DATA[t.id] = { rep:t.rep, states:t.states,
        pipeline:t.livePipeline||t.pipeline, op:t.op,
        color:t.color, attainment:t.attainment, dealCount:t.dealCount||0 };
    });
    STATE_TO_TERRITORY = {};
    Object.entries(TERRITORY_DATA).forEach(function(e){
      e[1].states.forEach(function(s){ STATE_TO_TERRITORY[s]=e[0]; });
    });
    if (typeof buildUSMap === \'function\') buildUSMap();
    console.log(\'[APEX360] Territories loaded:\', data.territories.length);
  }).catch(function(e){ console.warn(\'[APEX360] Territory load failed:\', e); });
}

function loadPipeline(territory) {
  territory = territory || \'ALL\';
  var path = \'/pipeline\' + (territory !== \'ALL\' ? \'?territory=\' + territory : \'\');
  apiFetch(path).then(function(data) {
    if (!data.deals || !data.deals.length) return;
    pipelineDeals = data.deals.map(function(d) {
      return { account:d.name||d.account, rep:d.rep, territory:d.territory,
        amount:d.amount, weighted:d.weighted, stage:d.stage,
        prob:d.probability, closeDate:d.closeDate,
        laer:d.laer, gpo:d.gpo||\'None\', risk:d.risk,
        forecast:d.forecast, nextAction:d.nextAction,
        actionDate:d.actionDate, q3:d.q3 };
    });
    if (typeof renderPipeline === \'function\') renderPipeline(\'all\');
    if (typeof renderTop80 === \'function\') renderTop80();
    console.log(\'[APEX360] Pipeline loaded:\', pipelineDeals.length, \'deals\');
  }).catch(function(e){ console.warn(\'[APEX360] Pipeline load failed:\', e); });
}

function loadWinLoss() {
  apiFetch(\'/win-loss\').then(function(data) {
    if (!data.results) return;
    window.liveWinLoss = data;
    console.log(\'[APEX360] Win/Loss loaded:\', data.count, \'records\');
  }).catch(function(e){ console.warn(\'[APEX360] Win/Loss load failed:\', e); });
}

var _agentHistories = { surgical:[], pricing:[], travel:[] };

function sendMsg(agentType) {
  var inputMap = {surgical:\'product-input\', pricing:\'finance-input\', travel:\'trip-input\'};
  var chatMap  = {surgical:\'product-chat\',  pricing:\'finance-chat\',  travel:\'trip-chat\'};
  var input = document.getElementById(inputMap[agentType] || \'product-input\');
  var chat  = document.getElementById(chatMap[agentType]  || \'product-chat\');
  if (!input || !chat || !input.value.trim()) return;
  var message = input.value.trim();
  input.value = \'\';
  var userBubble = document.createElement(\'div\');
  userBubble.className = \'msg msg-user\';
  userBubble.innerHTML = \'<div class="msg-bubble">\' + message + \'</div>\';
  chat.appendChild(userBubble);
  var typing = document.createElement(\'div\');
  typing.id = \'typing-\' + agentType;
  typing.className = \'msg\';
  typing.innerHTML = \'<div class="msg-avatar">AI</div><div class="msg-bubble" style="opacity:0.6">Thinking...</div>\';
  chat.appendChild(typing);
  chat.scrollTop = chat.scrollHeight;
  apiFetch(\'/agent/\' + agentType, {
    method: \'POST\',
    body: JSON.stringify({ message:message, history:_agentHistories[agentType]||[] })
  }).then(function(data) {
    var t = document.getElementById(\'typing-\' + agentType);
    if (t) t.remove();
    var response = data.response || \'No response.\';
    var botBubble = document.createElement(\'div\');
    botBubble.className = \'msg\';
    botBubble.innerHTML = \'<div class="msg-avatar">AI</div><div>\' +
      \'<div class="msg-bubble">\' + response.replace(/\\n/g,\'<br>\') + \'</div>\' +
      \'<div class="msg-time">Live - AWS Bedrock Claude</div></div>\';
    chat.appendChild(botBubble);
    chat.scrollTop = chat.scrollHeight;
    (_agentHistories[agentType] = _agentHistories[agentType] || []).push(
      {role:\'user\',content:message},
      {role:\'assistant\',content:response}
    );
    if (_agentHistories[agentType].length > 40)
      _agentHistories[agentType] = _agentHistories[agentType].slice(-40);
  }).catch(function(err) {
    var t = document.getElementById(\'typing-\' + agentType);
    if (t) t.remove();
    var e = document.createElement(\'div\');
    e.className = \'msg\';
    e.innerHTML = \'<div class="msg-avatar">!</div><div class="msg-bubble" style="color:#c23934">Error: \' + err.message + \'</div>\';
    chat.appendChild(e);
    chat.scrollTop = chat.scrollHeight;
  });
}

function askAgent(agentType, message) {
  var inputMap = {surgical:\'product-input\', pricing:\'finance-input\', travel:\'trip-input\'};
  var input = document.getElementById(inputMap[agentType] || \'product-input\');
  if (input) { input.value = message; sendMsg(agentType); }
}

setTimeout(function() {
  console.log(\'[APEX360] Live API:\', API_BASE);
  loadTerritories();
  loadPipeline(\'ALL\');
  loadWinLoss();
}, 800);

setTimeout(function() {
  var hdr = document.querySelector(\'.header-right\');
  if (hdr) {
    var badge = document.createElement(\'div\');
    badge.className = \'hdr-badge\';
    badge.style.background = \'#2e844a\';
    badge.title = \'API: \' + API_BASE;
    badge.textContent = \'Live API\';
    hdr.prepend(badge);
  }
}, 500);
"""

# Substitute the actual API URL
live_script = live_script.replace("__API_URL__", api_url)

# Inject before the last </script> tag
injection_point = html.rfind("</script>")
if injection_point == -1:
    print("ERROR: Could not find </script> tag in HTML file")
else:
    patched = html[:injection_point] + live_script + "\n</script>" + html[injection_point+9:]

    # Update page title
    patched = patched.replace(
        "<title>APEX360 | VitalEdge Commercial Command Center</title>",
        "<title>APEX360 | VitalEdge -- Live</title>"
    )

    out = "apex360_live.html"
    with open(out, "w", encoding="utf-8") as f:
        f.write(patched)

    CONFIG["live_html"] = out
    print(f"Saved:    {out} ({len(patched):,} chars)")
    print()
    print("What was injected:")
    print("  - API_BASE set to your live API Gateway URL")
    print("  - loadTerritories() rebuilds the US map from live data")
    print("  - loadPipeline() replaces hardcoded deals with real data")
    print("  - sendMsg() routes agent chats to Bedrock Claude")
    print("  - Green Live API badge added to header")


---
## Cell 8 -- Upload Dashboard to S3

Uploads `apex360_live.html` as `index.html` to the public app bucket.
This is your live, shareable demo URL.

**To redeploy:** Edit HTML, re-run Cell 7 to patch it, re-run this cell.
Changes are live immediately -- no cache to invalidate.


In [ ]:
live_path = CONFIG.get("live_html", "apex360_live.html")

with open(live_path, "rb") as f:
    html_bytes = f.read()

# Upload as index.html with no-cache so users always get the latest version
s3.put_object(
    Bucket=CONFIG["app_bucket"],
    Key="index.html",
    Body=html_bytes,
    ContentType="text/html",
    CacheControl="no-cache, no-store, must-revalidate"
)

CONFIG["app_url"] = (
    f"http://{CONFIG['app_bucket']}"
    f".s3-website-{CONFIG['region']}.amazonaws.com"
)

print(f"  Uploaded: {len(html_bytes):,} bytes")
print()
print("=" * 62)
print("  YOUR LIVE DASHBOARD:")
print()
print(f"  {CONFIG['app_url']}")
print()
print("=" * 62)
print()
print("  Open in any browser. The green 'Live API' badge confirms")
print("  real data is loading from your AWS API Gateway.")


---
## Cell 9 -- Bedrock Knowledge Bases (Optional)

**Skip this for now.** The agents work perfectly from their system prompts.

Run this cell only when you have real documents to upload:
- APEX product spec sheets
- Clinical outcomes studies
- Competitive battlecards vs Intuitive/Stryker/Medtronic
- GPO contract PDFs
- ROI model documentation

**Cost when running:** ~$90/month (OpenSearch Serverless)
**Cost when deleted:** $0

**Workflow:** Deploy KBs morning of an interview, demo, delete the same day = ~$3 cost.

**Time:** 10-15 minutes (OpenSearch collection provisioning)


In [ ]:
# SKIP THIS CELL FOR THE PROTOTYPE
# Uncomment and run only when you have documents to upload

# bedrock_agent = boto3.client("bedrock-agent", region_name=CONFIG["region"])
#
# kb_trust = {
#     "Version": "2012-10-17",
#     "Statement": [{"Effect": "Allow",
#         "Principal": {"Service": "bedrock.amazonaws.com"},
#         "Action": "sts:AssumeRole",
#         "Condition": {"StringEquals": {"aws:SourceAccount": CONFIG["account_id"]}}}]
# }
# try:
#     kb_role_arn = iam.create_role(
#         RoleName=CONFIG["kb_role_name"],
#         AssumeRolePolicyDocument=json.dumps(kb_trust),
#         Description="Bedrock KB role for APEX360"
#     )["Role"]["Arn"]
# except iam.exceptions.EntityAlreadyExistsException:
#     kb_role_arn = iam.get_role(RoleName=CONFIG["kb_role_name"])["Role"]["Arn"]
#
# for p in ["arn:aws:iam::aws:policy/AmazonS3ReadOnlyAccess",
#           "arn:aws:iam::aws:policy/AmazonBedrockFullAccess",
#           "arn:aws:iam::aws:policy/AmazonOpenSearchServiceFullAccess"]:
#     try: iam.attach_role_policy(RoleName=CONFIG["kb_role_name"], PolicyArn=p)
#     except: pass
#
# CONFIG["kb_role_arn"] = kb_role_arn
# time.sleep(12)
#
# def create_kb(name, description, prefix):
#     print(f"Creating {name}...")
#     kb = bedrock_agent.create_knowledge_base(
#         name=name, description=description, roleArn=kb_role_arn,
#         knowledgeBaseConfiguration={"type":"VECTOR",
#             "vectorKnowledgeBaseConfiguration":{"embeddingModelArn":
#                 f"arn:aws:bedrock:{CONFIG['region']}::foundation-model/amazon.titan-embed-text-v2:0"}},
#         storageConfiguration={"type":"OPENSEARCH_SERVERLESS",
#             "opensearchServerlessConfiguration":{"collectionArn":"",
#                 "vectorIndexName":f"apex360-{name.split('-')[1]}-idx",
#                 "fieldMapping":{"vectorField":"embedding","textField":"text","metadataField":"metadata"}}}
#     )
#     kb_id = kb["knowledgeBase"]["knowledgeBaseId"]
#     for _ in range(24):
#         status = bedrock_agent.get_knowledge_base(knowledgeBaseId=kb_id)["knowledgeBase"]["status"]
#         if status == "ACTIVE": print(f"  ACTIVE: {kb_id}"); break
#         print(f"  {status}..."); time.sleep(15)
#     ds = bedrock_agent.create_data_source(
#         knowledgeBaseId=kb_id, name=f"{name}-s3",
#         dataSourceConfiguration={"type":"S3","s3Configuration":{
#             "bucketArn":f"arn:aws:s3:::{CONFIG['kb_docs_bucket']}",
#             "inclusionPrefixes":[prefix]}},
#         vectorIngestionConfiguration={"chunkingConfiguration":{
#             "chunkingStrategy":"SEMANTIC",
#             "semanticChunkingConfiguration":{"maxTokens":300,"bufferSize":1,
#                 "breakpointPercentileThreshold":95}}})
#     ds_id = ds["dataSource"]["dataSourceId"]
#     return kb_id, ds_id
#
# surgical_kb, surgical_ds = create_kb("apex360-surgical-kb",
#     "APEX surgical robotics specs, clinical outcomes, competitive battlecards", "surgical/")
# pricing_kb, pricing_ds = create_kb("apex360-pricing-kb",
#     "GPO contracts, pricing guides, ROI models, discount policies", "pricing/")
# travel_kb, travel_ds = create_kb("apex360-travel-kb",
#     "Territory guidelines, account priorities, travel policies", "travel/")
#
# CONFIG["kb_ids"] = {
#     "surgical": {"kb_id": surgical_kb, "ds_id": surgical_ds},
#     "pricing":  {"kb_id": pricing_kb,  "ds_id": pricing_ds},
#     "travel":   {"kb_id": travel_kb,   "ds_id": travel_ds},
# }
# print("Knowledge Bases created. Upload PDFs to trigger ingestion.")
# print(f"  s3://{CONFIG['kb_docs_bucket']}/surgical/spec_sheets/your_file.pdf")

print("Cell 9 is commented out -- skip for prototype.")
print("Run this cell when you have real documents to load into the agents.")


---
## Cell 10 -- Smoke Test

Hits every API endpoint and confirms it returns data.

**If any endpoint fails:** Go to AWS Console -> CloudWatch -> Log groups ->
`/aws/lambda/apex360-{name}` and check the most recent log stream.


In [ ]:
api = CONFIG.get("api_url", "")
print(f"Testing: {api}")
print("-" * 62)
time.sleep(5)  # Let deployment settle

tests = [
    ("Territories",              f"{api}/territories",                       "territories"),
    ("Pipeline -- all",          f"{api}/pipeline",                          "deals"),
    ("Pipeline -- NW-1",         f"{api}/pipeline?territory=NW-1",           "deals"),
    ("Accounts -- all",          f"{api}/accounts",                          "accounts"),
    ("Accounts -- ACC-001",      f"{api}/accounts?id=ACC-001",               "accounts"),
    ("Installed Base ACC-001",   f"{api}/installed-base?account_id=ACC-001", "devices"),
    ("Win/Loss -- all",          f"{api}/win-loss",                          "results"),
    ("Win/Loss -- wins",         f"{api}/win-loss?outcome=WIN",              "results"),
    ("Win/Loss -- Intuitive",    f"{api}/win-loss?competitor=Intuitive",     "results"),
]

all_ok = True
for label, url, key in tests:
    try:
        with urllib.request.urlopen(url, timeout=15) as r:
            data = json.loads(r.read())
            count = data.get("count") or len(data.get(key, []))
            extra = ""
            if "total" in data: extra = f"  ${data['total']/1e6:.1f}M"
            if "winRate" in data: extra = f"  {data['winRate']}% win rate"
            print(f"  OK    {label:<30} {count:>4} records{extra}")
    except Exception as e:
        print(f"  FAIL  {label:<30} {str(e)[:50]}")
        all_ok = False

print()
# Test agent
try:
    req = urllib.request.Request(
        f"{api}/agent/surgical",
        data=json.dumps({
            "message": "What are APEX's key advantages vs Intuitive da Vinci in a competitive eval?",
            "history": []
        }).encode(),
        headers={"Content-Type": "application/json"},
        method="POST"
    )
    with urllib.request.urlopen(req, timeout=30) as r:
        data = json.loads(r.read())
        preview = data.get("response", "")[:100]
        print(f"  OK    {'Surgical Agent':<30} '{preview}...'")
except Exception as e:
    print(f"  FAIL  {'Surgical Agent':<30} {str(e)[:60]}")
    all_ok = False

print()
print("-" * 62)
if all_ok:
    print("  ALL TESTS PASSED")
else:
    print("  SOME TESTS FAILED -- check CloudWatch logs")
    print("  AWS Console -> CloudWatch -> Log groups -> /aws/lambda/apex360-*")


## Cell 11 -- Deployment Summary

In [ ]:
print("=" * 65)
print("  APEX360 VitalEdge -- DEPLOYMENT COMPLETE")
print("=" * 65)
print()
print("  LIVE DASHBOARD:")
print(f"    {CONFIG.get('app_url', 'run Cell 8')}")
print()
print("  API ENDPOINTS:")
api = CONFIG.get("api_url", "https://YOUR_API")
for method, path, desc in [
    ("GET",  "/territories",                        "All 15 territories + live pipeline rollup"),
    ("GET",  "/pipeline",                           "All deals (optional ?territory=NW-1)"),
    ("GET",  "/accounts",                           "All accounts with installed base summary"),
    ("GET",  "/accounts?id=ACC-001",                "Single account detail + device inventory"),
    ("GET",  "/installed-base?account_id=ACC-001",  "Full device inventory with model data"),
    ("GET",  "/win-loss",                           "W/L records (optional ?outcome=WIN)"),
    ("GET",  "/win-loss?competitor=Intuitive",      "Filter W/L by competitor"),
    ("POST", "/agent/surgical",                     "APEX Surgical Intelligence Agent"),
    ("POST", "/agent/pricing",                      "Contract and Pricing Agent"),
    ("POST", "/agent/travel",                       "Travel Optimization Agent"),
]:
    print(f"    {method:<5} {api}{path}")
    print(f"           -> {desc}")
print()
print("  DATA IN S3:")
print(f"    s3://{CONFIG['data_bucket']}/pipeline/         (97 deals)")
print(f"    s3://{CONFIG['data_bucket']}/accounts/         (55 accounts, 292 contacts)")
print(f"    s3://{CONFIG['data_bucket']}/installed_base/   (364 devices, 18 models)")
print(f"    s3://{CONFIG['data_bucket']}/contracts/        (55 contracts)")
print(f"    s3://{CONFIG['data_bucket']}/win_loss/         (85 records)")
print(f"    s3://{CONFIG['data_bucket']}/activities/       (220 activities)")
print()
if CONFIG.get("cognito_pool_id"):
    print("  COGNITO:")
    print(f"    Pool ID:   {CONFIG['cognito_pool_id']}")
    print(f"    Client ID: {CONFIG['cognito_client_id']}")
    print(f"    AVP login: {CONFIG['admin_email']} / APEX360!Avp2025")
    print()
print("  INTERVIEW TOUR (AWS Console):")
for step in [
    "1.  Dashboard URL in browser -- show Live API badge + real data loading",
    "2.  S3 -> apex360-data -> show 13 CSVs organized by domain",
    "3.  Lambda -> filter 'apex360' -> show 6 functions, click one to show code",
    "4.  API Gateway -> apex360-api -> show routes, click Test on /territories",
    "5.  Cognito -> apex360-users -> show 17 accounts with territory_id claims",
    "6.  Dashboard -> click a territory on the map -> show filtered pipeline",
    "7.  Dashboard -> GTM Intelligence -> Opportunity Engine -> show scoring",
    "8.  Dashboard -> Surgical Agent -> ask a competitive question -> show Claude",
    "9.  Explain: 'The JWT token IS the security boundary -- Lambda enforces it'",
]:
    print(f"    {step}")
print()
print("  MONTHLY COST:")
for svc, cost in [
    ("S3 (3 buckets, ~2MB data)",            "~$0.01"),
    ("Lambda (light traffic)",               "~$0.00"),
    ("API Gateway",                          "~$1.00"),
    ("Bedrock Claude (agent calls)",         "~$15.00"),
    ("Cognito (under 50K MAU)",              "~$0.00"),
    ("OpenSearch KB (if Cell 9 run)",        "~$90.00"),
]:
    print(f"    {svc:<44} {cost}/month")
print()
print("    Without Bedrock KBs:  ~$16/month")
print("    With Bedrock KBs:     ~$106/month")
print("    KBs billed when running -- delete after demo to stop charges")
print("=" * 65)


---
## Cell 12 -- Teardown

**Run this to delete all APEX360 AWS resources.**
Everything is commented out -- nothing runs by accident.
Run teardown in the same session as deployment (while CONFIG is populated).

**Most common use:** Uncomment only the Bedrock KB section to stop the
~$90/month OpenSearch charge after a demo, while keeping everything else running.


In [ ]:
# TEARDOWN -- uncomment sections to delete specific resources

# ---- Lambda functions ------------------------------------------------
# lam = boto3.client("lambda", region_name=CONFIG["region"])
# for fn in ["apex360-pipeline","apex360-accounts","apex360-territories",
#            "apex360-installed-base","apex360-win-loss","apex360-agent",
#            "apex360-kb-sync"]:
#     try: lam.delete_function(FunctionName=fn); print(f"Deleted Lambda: {fn}")
#     except Exception as e: print(f"  {fn}: {e}")

# ---- API Gateway ----------------------------------------------------
# apigw = boto3.client("apigateway", region_name=CONFIG["region"])
# for a in apigw.get_rest_apis()["items"]:
#     if a["name"] == "apex360-api":
#         apigw.delete_rest_api(restApiId=a["id"])
#         print(f"Deleted API: {a['id']}")

# ---- Bedrock Knowledge Bases (uncomment to stop $90/month charge) ---
# ba = boto3.client("bedrock-agent", region_name=CONFIG["region"])
# for t, ids in CONFIG.get("kb_ids", {}).items():
#     kb_id = ids.get("kb_id")
#     if not kb_id: continue
#     try: ba.delete_data_source(knowledgeBaseId=kb_id, dataSourceId=ids["ds_id"])
#     except: pass
#     try: ba.delete_knowledge_base(knowledgeBaseId=kb_id); print(f"Deleted KB: {t}")
#     except Exception as e: print(f"  KB {t}: {e}")

# ---- Cognito --------------------------------------------------------
# cog = boto3.client("cognito-idp", region_name=CONFIG["region"])
# pid = CONFIG.get("cognito_pool_id")
# if pid:
#     for c in cog.list_user_pool_clients(
#             UserPoolId=pid, MaxResults=10).get("UserPoolClients", []):
#         cog.delete_user_pool_client(UserPoolId=pid, ClientId=c["ClientId"])
#     cog.delete_user_pool(UserPoolId=pid)
#     print(f"Deleted Cognito pool: {pid}")

# ---- S3 Buckets -----------------------------------------------------
# s3r = boto3.resource("s3", region_name=CONFIG["region"])
# for bname in [CONFIG["data_bucket"], CONFIG["app_bucket"], CONFIG["kb_docs_bucket"]]:
#     try:
#         b = s3r.Bucket(bname)
#         b.object_versions.delete()
#         b.objects.all().delete()
#         b.delete()
#         print(f"Deleted bucket: {bname}")
#     except Exception as e:
#         print(f"  {bname}: {e}")

# ---- IAM Roles -------------------------------------------------------
# for rname in [CONFIG["lambda_role_name"], CONFIG["kb_role_name"]]:
#     try:
#         for p in iam.list_attached_role_policies(RoleName=rname)["AttachedPolicies"]:
#             iam.detach_role_policy(RoleName=rname, PolicyArn=p["PolicyArn"])
#         iam.delete_role(RoleName=rname)
#         print(f"Deleted IAM role: {rname}")
#     except Exception as e:
#         print(f"  {rname}: {e}")

print("Teardown is commented out for safety.")
print("Uncomment the section(s) above you want to delete and re-run.")
print()
print("Most common: uncomment only the Bedrock KB section")
print("to stop the ~$90/month OpenSearch charge after a demo.")
